## 1. Motivation and model overview

A player in possession must decide not only whether to pass, but **where to direct the ball**. Pass selection models this decision as a probability distribution over potential destinations, conditional on attempting a pass. It is distinct from *pass probability*, which estimates whether a pass to a specified destination will be completed. A destination can be attractive without being safe, and a safe destination need not be frequently selected. These components describe different aspects of the same decision.

The distinction is central to spatial models of football. Spearman's *Beyond Expected Goals* [@spearman2018beyond] places a pass selection model alongside control and scoring opportunity in a framework for evaluating off-ball attacking opportunities. Fernández, Bornn and Cervone [@fernandez2021] explicitly model pass selection as a normalized spatial probability surface within an expected-possession-value framework. PRESS follows this probabilistic interpretation of destination choice, while learning the utility of each destination using gradient-boosted trees instead of a convolutional network.

At each observed passing decision, the pitch is discretized into candidate destinations on a **one-metre grid**. The observed endpoint is mapped to its nearest cell centre. The resulting probabilities sum to one across the complete candidate set; they are conditional on a pass being attempted, irrespective of whether the observed pass was successful.

Context-label meanings and the `previous_action` convention are defined in the [User guide](../setup/possession_and_checkpoint_setup.ipynb#context-labels).


## 2. Probabilistic destination-choice model

Let $s_i$ denote the tracked game state at passing decision $i$, and let $\mathcal G_i$ be its set of candidate destination cells. Each cell $j$ is represented by features $z_{ij}$ describing the pass and the current spatial configuration. A **single shared function** assigns a utility to each destination:

$$
u_{ij}=f_\theta(z_{ij}).$$

PRESS represents $f_\theta$ as an additive ensemble of XGBoost regression trees. Utilities express relative preference, not pass-completion probabilities. The multinomial-logit model converts them into a categorical distribution:

$$
\boxed{p_{ij}=P(Y_i=j\mid s_i,\text{pass})=
\frac{\exp(u_{ij})}{\sum_{k\in\mathcal G_i}\exp(u_{ik})}.}
$$

The normalization couples every candidate in the same decision: changing the utility of one destination changes the probabilities of the others. No separate binary classifier is trained for each cell.

The ensemble is estimated from observed destinations by minimizing the categorical negative log-likelihood,

$$\mathcal L(\theta)=-\sum_i\log p_{i,y_i}(\theta),$$

where $y_i$ is the observed destination cell. The [benchmark](../../benchmarks/03_pass_selection.qmd) describes the boosting procedure, regularization, model selection and validation.

After fitting, a scalar temperature $T$ calibrates the predicted distribution,

$$p_{ij}^{(T)}=\frac{p_{ij}^{1/T}}{\sum_k p_{ik}^{1/T}},$$

without changing the ordering of destinations. The installed model uses the flat multinomial-logit specification; an exploratory cross-nested extension is evaluated separately in the benchmark.

## 3. Model features

The utility model uses **19 numerical predictors** and two categorical context variables. All are computed from information available at the passing decision; the realized receiver, pass outcome and subsequent events are excluded.

| Feature group | Predictors | Interpretation |
|---|---|---|
| Pass geometry and carrier motion | Pass distance; signed pass angle relative to carrier velocity; carrier speed | Spatial and kinematic properties of the candidate pass |
| Pitch control | Raw attacking pitch control at the target | Availability of the destination to the attacking team |
| Expected threat (xT) | Own-team and opponent xT at the carrier and target | Spatial estimates of the value of ball possession at the current and candidate locations |
| Defensive structure | Own and opponent pressure-line indices at the carrier and target | Position relative to three ordered longitudinal line centres, following Fernández et al. (2021) |
| Ball retention | One-second ball-drive loss probability | Risk associated with retaining the ball instead of passing |
| Match context | Match minute; score difference known at the decision | Temporal and match-state context |
| Player positioning | Numbers of teammates and opponents beyond the transverse line through the carrier and through the target | Current team and opponent distribution in the attacking direction |
| Categorical context | Possession start type; previous action | Event-based context, one-hot encoded using the vocabulary learned during fitting |

For the four player counts, *beyond* means further toward the **carrier team's opposing goal**, using that attacking direction for both rosters. The carrier is excluded from teammate counts, tracked goalkeepers are retained, and players exactly on the transverse line are not counted. Counts are evaluated separately at the carrier and at each candidate target; they are not restricted to the segment between them. Pressure-line indices identify the nearest of three ordered line centres. Pitch control is used in its raw, uncalibrated form for the choice utility.

**Expected threat.** The xT features follow the expected-threat framework introduced by Singh [@singh2018xt]. They describe the spatial value of possession from each team's perspective, evaluated both at the carrier and at the proposed destination. The calculation used in PRESS is documented in the [Expected Threat vignette](../extras/expected_threat/11.expected_threat.ipynb).

**Pressure lines.** Following the spatial line representation used by Fernández, Bornn and Cervone [@fernandez2021], the model characterizes the positions of the teams' longitudinal lines. For each team, the pressure-line index records which of three ordered line centres is closest to the carrier or candidate destination. This captures whether a potential pass moves the ball across the teams' positional structure.

**Event context.** *Possession start type* is derived from SkillCorner's dynamical events and identifies how the action began, for example through a pass or an interception. *Previous action* identifies the immediately preceding segment: it can be a ball drive or the start type of the action when no preceding ball-drive segment is defined. These labels follow the segmentation conventions described in [Possessions and checkpoints](../setup/possession_and_checkpoint_setup.ipynb).

## 4. Feature importance

The following results summarize **training split-gain importance** for the fitted full-data model, together with each feature's rank across the five held-out evaluation fits. Gain is an attribution of the ensemble's split improvements, not a causal effect or a measure of how predictions would change if a feature were removed.

| Feature | Full-data gain share | Rank across five folds |
|---|---:|---:|
| Pass distance | 34.44% | 1–1 |
| Raw target pitch control | 20.91% | 2–2 |
| Pass angle | 14.41% | 3–3 |
| Teammates beyond target | 5.72% | 4–4 |
| Opponents beyond target | 5.04% | 5–7 |
| Carrier speed | 3.49% | 5–6 |

Distance, pitch control and angle account for **69.76%** of full-data split gain (rounded component shares) and occupy the same first three positions in every fold. This is consistent with geometry and destination accessibility being particularly informative for observed pass choice.

![Training split-gain allocation for the fitted model and five evaluation folds.](../../figures/feature_importance_pass_selection.png)

## 5. Application to a game state

Brisbane FC–Perth Glory: N. Acosta chooses and completes a pass to J. Hingert (checkpoint 54295). Set `SKILLCORNER_OPENDATA` to your local open-data repository ([Data setup](../../data.qmd)). The dashed line shows the matched destination. Colors display relative preference: 100% is the most probable destination cell. The stored cell probabilities remain a distribution summing to one.

The action labels are retrieved with `get_frame_context`; the match minute is calculated from the timestamp returned by `get_frame`. The carrier-team score difference is read from the current possession's dynamical event. It is supplied explicitly because these processed tracking rows do not contain score information.


In [7]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt
import jax

jax.config.update("jax_enable_x64", True)
from PRESS import PRESS

opendata = Path(os.environ.get("SKILLCORNER_OPENDATA", Path.cwd().parent / "opendata"))
match_dir = opendata / "data/processed/2026-10-15_training_v5_outfield/matches/1925299"
press = PRESS.from_skillcorner(
    match_dir, actions_path=match_dir / "1925299_action_selection_base.parquet"
)
frame_id = 54295

from PRESS.model import PassSelectionModel
from PRESS.visualization import plot_pass_map

choice = PassSelectionModel.pretrained()
context = press.get_frame_context(frame_id)
frame = press.get_frame(frame_id).iloc[0]

# Match minute comes from the observed tracking timestamp.
match_clock_min = pd.to_timedelta(frame.timestamp).total_seconds() / 60

# The context getter supplies action labels, not the score. Read the known
# score from the dynamical event of the current individual possession.
possessions = press.get_player_possessions()
current_possession = possessions.loc[
    possessions.frame_start.le(frame_id) & possessions.frame_end.ge(frame_id)
].iloc[0]
events = pd.read_csv(
    opendata / "data/matches/1925299/1925299_dynamic_events.csv", low_memory=False,
)
origin = events.loc[
    events.event_type.eq("player_possession")
    & events.event_id.eq(current_possession.possession_id)
].iloc[0]
score_difference = float(origin.team_score - origin.opponent_team_score)

press.compute_ppcf(frame_id)
press.compute_ball_drive_risk(frame_id)
press.compute_pass_selection(
    frame_id, choice.xt_grid,
    start_type_raw=context["start_type"], previous_action_raw=context["previous_action"],
    match_clock_min=match_clock_min, score_difference_prefix=score_difference,
)
plot_pass_map(press, frame_id, field="pass_selection", pass_target=(-12.15, -28.05))
plt.savefig("figures/vignette_03_pass_selection_pitch.png", dpi=170)
plt.savefig("figures/vignette_03_pass_selection_pitch.svg")
plt.close()


![Pass selection on the real checkpoint; color intensity is relative to the most probable cell.](../../figures/vignette_03_pass_selection_pitch.png)

## 6. Using the fitted model

The model can also be loaded directly, with its saved feature vocabulary and fitted physical-model dependencies:

```python
from PRESS.model import PassSelectionModel
choice = PassSelectionModel.pretrained()
press = PRESS(tracking, pass_selection_model=choice)
press.compute_ppcf(frame_id)
press.compute_ball_drive_risk(frame_id, t_press=1.0)
probability = press.compute_pass_selection(
    frame_id, xT_grid,
    start_type_raw=context["start_type"], previous_action_raw=context["previous_action"],
    match_clock_min=match_clock_min, score_difference_prefix=score_difference,
)
```

Here `context`, `match_clock_min` and `score_difference` are the observed values recovered in the preceding example; retrieve them again when selecting another frame. The positional `xT_grid` argument remains for API compatibility; the installed choice model retains its fitted xT representation. The [benchmark](../../benchmarks/03_pass_selection.qmd) reports the evaluation protocol, training details, exploratory nested alternative and comparison with Fernández et al. [@fernandez2021].